# CNN Transfer Learning — Intel Image Classification

## Overview

This notebook demonstrates **Transfer Learning** with a **Convolutional Neural Network (CNN)** using PyTorch on the [Intel Image Classification](https://www.kaggle.com/datasets/puneet6060/intel-image-classification) dataset.

### Dataset
- **6 classes**: buildings, forest, glacier, mountain, sea, street
- **~14,000** training images and **~3,000** test images
- Images are **150×150** RGB JPEGs of natural scenes

### What is Transfer Learning?

Instead of training a CNN from scratch, we take a model (ResNet-18) that was **pre-trained on ImageNet** (1.2 million images, 1000 classes) and adapt it to our task:

1. **Feature Extraction** — Freeze the pre-trained backbone and only train a new classification head. The convolutional layers already know how to extract edges, textures, shapes, and objects.
2. **Fine-Tuning** — Unfreeze some of the deeper layers and train them with a small learning rate so the model can adapt its learned representations to our specific domain.

### Why Transfer Learning?
- Much faster convergence (fewer epochs needed)
- Works well with smaller datasets
- Leverages powerful features learned from millions of images

### Notebook Outline
1. Setup & Imports
2. Data Exploration
3. Data Loading & Augmentation
4. Model Architecture (ResNet-18)
5. Phase 1 — Feature Extraction (frozen backbone)
6. Phase 2 — Fine-Tuning (unfrozen layers)
7. Evaluation on Test Set
8. Predictions & Error Analysis
9. Save Model

---
## 1. Setup & Imports

In [ ]:
import os
import sys
import warnings

import numpy as np
import torch
import torch.nn as nn
import matplotlib.pyplot as plt
import seaborn as sns

warnings.filterwarnings("ignore")
sns.set_theme(style="whitegrid", palette="Set2")

sys.path.insert(0, os.path.dirname(os.path.abspath("__file__")))

from data_loader import (
    load_datasets,
    split_train_val,
    create_dataloaders,
    denormalize,
    CLASS_NAMES,
    NUM_CLASSES,
    TRAIN_DIR,
    TEST_DIR,
)
from model import (
    create_model,
    unfreeze_layers,
    get_trainable_params,
    train_model,
    evaluate,
    predict_single,
    save_model,
)
from visualization import (
    plot_class_distribution,
    plot_sample_images,
    plot_augmented_samples,
    plot_training_history,
    plot_lr_schedule,
    plot_confusion_matrix,
    print_classification_report,
    plot_predictions_grid,
    plot_top_losses,
)

RANDOM_STATE = 42
np.random.seed(RANDOM_STATE)
torch.manual_seed(RANDOM_STATE)

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"PyTorch {torch.__version__}  |  Device: {device}")

---
## 2. Data Exploration

In [ ]:
train_dataset, test_dataset = load_datasets()

print(f"Training samples : {len(train_dataset):,}")
print(f"Test samples     : {len(test_dataset):,}")
print(f"Classes          : {CLASS_NAMES}")
print(f"Number of classes: {NUM_CLASSES}")

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(14, 4))
plot_class_distribution(train_dataset, title="Training Set Distribution", ax=axes[0])
plot_class_distribution(test_dataset, title="Test Set Distribution", ax=axes[1])
plt.tight_layout()
plt.show()

In [ ]:
fig = plot_sample_images(train_dataset, n_per_class=4)
plt.show()

---
## 3. Data Loading & Augmentation

We split the training data into **train (85%)** and **validation (15%)** subsets. Data augmentation (random flips, rotations, color jitter) is applied only to the training set to reduce overfitting.

In [ ]:
train_subset, val_subset = split_train_val(train_dataset, val_ratio=0.15, seed=RANDOM_STATE)

print(f"Train subset: {len(train_subset):,}")
print(f"Val subset  : {len(val_subset):,}")
print(f"Test set    : {len(test_dataset):,}")

In [ ]:
BATCH_SIZE = 32

train_loader, val_loader, test_loader = create_dataloaders(
    train_subset, val_subset, test_dataset,
    batch_size=BATCH_SIZE, num_workers=2,
)

print(f"Train batches: {len(train_loader)}")
print(f"Val batches  : {len(val_loader)}")
print(f"Test batches : {len(test_loader)}")

In [ ]:
fig = plot_augmented_samples(train_dataset, idx=0, n_samples=6)
plt.show()

---
## 4. Model Architecture — ResNet-18

We use a **ResNet-18** pre-trained on ImageNet as our backbone. The original final fully-connected layer (1000 classes) is replaced with a new head for our 6 classes.

**Phase 1 (Feature Extraction):** All backbone layers are frozen — only the new classifier trains.  
**Phase 2 (Fine-Tuning):** We unfreeze the last few residual blocks and train with a smaller learning rate.

In [ ]:
model = create_model(num_classes=NUM_CLASSES, pretrained=True, freeze_backbone=True)
model = model.to(device)

total_params = sum(p.numel() for p in model.parameters())
trainable, _ = get_trainable_params(model)

print(f"Total parameters    : {total_params:,}")
print(f"Trainable parameters: {trainable:,}")
print(f"Frozen parameters   : {total_params - trainable:,}")
print(f"\nClassifier head:\n{model.fc}")

---
## 5. Phase 1 — Feature Extraction (Frozen Backbone)

Train only the classification head with a moderate learning rate. This is fast since gradients don't flow through the backbone.

In [ ]:
criterion = nn.CrossEntropyLoss()

optimizer_phase1 = torch.optim.Adam(model.fc.parameters(), lr=1e-3, weight_decay=1e-4)
scheduler_phase1 = torch.optim.lr_scheduler.ReduceLROnPlateau(
    optimizer_phase1, mode="min", factor=0.5, patience=2, verbose=False,
)

EPOCHS_PHASE1 = 8

print(f"Phase 1: Training classifier head for {EPOCHS_PHASE1} epochs...\n")
model, history_phase1 = train_model(
    model, train_loader, val_loader, criterion,
    optimizer_phase1, scheduler_phase1,
    device=device, num_epochs=EPOCHS_PHASE1,
)

In [ ]:
fig = plot_training_history(history_phase1)
fig.suptitle("Phase 1 — Feature Extraction", fontsize=14, fontweight="bold", y=1.03)
plt.show()

---
## 6. Phase 2 — Fine-Tuning (Unfreezing Deeper Layers)

Now we unfreeze the last 2 residual blocks and continue training with a **lower learning rate** to gently adapt the pre-trained features.

In [ ]:
unfreeze_layers(model, num_layers=2)

trainable, trainable_params = get_trainable_params(model)
print(f"Trainable parameters after unfreezing: {trainable:,}")

In [ ]:
optimizer_phase2 = torch.optim.Adam([
    {"params": [p for n, p in model.named_parameters()
                if p.requires_grad and "fc" not in n], "lr": 1e-4},
    {"params": model.fc.parameters(), "lr": 5e-4},
], weight_decay=1e-4)

scheduler_phase2 = torch.optim.lr_scheduler.ReduceLROnPlateau(
    optimizer_phase2, mode="min", factor=0.5, patience=2, verbose=False,
)

EPOCHS_PHASE2 = 7

print(f"Phase 2: Fine-tuning for {EPOCHS_PHASE2} epochs...\n")
model, history_phase2 = train_model(
    model, train_loader, val_loader, criterion,
    optimizer_phase2, scheduler_phase2,
    device=device, num_epochs=EPOCHS_PHASE2,
)

In [ ]:
fig = plot_training_history(history_phase2)
fig.suptitle("Phase 2 — Fine-Tuning", fontsize=14, fontweight="bold", y=1.03)
plt.show()

In [ ]:
combined_history = {
    k: history_phase1[k] + history_phase2[k] for k in history_phase1
}
fig = plot_training_history(combined_history)
fig.suptitle("Full Training History (Phase 1 + Phase 2)", fontsize=14, fontweight="bold", y=1.03)
plt.axvline(x=EPOCHS_PHASE1, color="gray", linestyle="--", alpha=0.6)
plt.show()

---
## 7. Evaluation on Test Set

In [ ]:
y_pred, y_true = evaluate(model, test_loader, device)

test_acc = sum(p == t for p, t in zip(y_pred, y_true)) / len(y_true)
print(f"Test Accuracy: {test_acc:.4f}\n")

print_classification_report(y_true, y_pred)

In [ ]:
plot_confusion_matrix(y_true, y_pred)
plt.show()

---
## 8. Predictions & Error Analysis

In [ ]:
fig = plot_predictions_grid(model, test_dataset, device, n_samples=12, ncols=4)
plt.show()

In [ ]:
fig = plot_top_losses(model, test_dataset, device, n=8)
plt.show()

### Single Image Prediction

In [ ]:
sample_img, sample_label = test_dataset[42]
pred_class, probs = predict_single(model, sample_img, device, CLASS_NAMES)

fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(10, 4))

img_display = denormalize(sample_img).permute(1, 2, 0).numpy()
ax1.imshow(img_display)
ax1.set_title(f"True: {CLASS_NAMES[sample_label]}  |  Pred: {pred_class}", fontsize=12)
ax1.axis("off")

colors = ["tab:blue" if i != probs.argmax().item() else "tab:red"
          for i in range(len(CLASS_NAMES))]
ax2.barh(CLASS_NAMES, probs.cpu().numpy(), color=colors)
ax2.set_xlabel("Probability")
ax2.set_title("Class Probabilities", fontsize=12)
ax2.set_xlim(0, 1)

plt.tight_layout()
plt.show()

---
## 9. Save Model

In [ ]:
model_path = os.path.join(os.path.dirname(os.path.abspath("__file__")), "resnet18_intel_scene.pth")
save_model(model, model_path)
print(f"Model saved to: {model_path}")

file_size_mb = os.path.getsize(model_path) / (1024 * 1024)
print(f"Model file size: {file_size_mb:.1f} MB")

---
## Summary

| Phase | Epochs | Strategy | What was trained |
|-------|--------|----------|------------------|
| 1 — Feature Extraction | 8 | Frozen backbone | Only the FC head |
| 2 — Fine-Tuning | 7 | Unfrozen last 2 blocks | FC head + last 2 residual blocks |

**Key Takeaways:**
- Transfer learning from ImageNet gives a strong starting point even for a different domain (natural scenes)
- Phase 1 quickly reaches good accuracy by leveraging pre-trained features
- Phase 2 fine-tuning further improves performance by adapting deeper features
- Data augmentation helps prevent overfitting on the relatively small training set
- The `ReduceLROnPlateau` scheduler automatically lowers the learning rate when validation loss plateaus